<a href="https://colab.research.google.com/github/abhisheklal-nasca/DigiYatra/blob/main/AI_School_Water_Consumption_Predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import gradio as gr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

In [2]:
import numpy as np
import pandas as pd


from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score


# --------------------------------
# 1. Generate sample training data
# --------------------------------


np.random.seed(42)


records = []


for day in range(1500):


    students = np.random.randint(300, 2501)
    staff = np.random.randint(20, 151)
    temperature = np.random.uniform(18, 43)
    event = np.random.randint(0, 2)


    # Calculate temperature effect
    temperature_factor = max(temperature - 25, 0)


    # Calculate daily water consumption
    daily_water = (
        students * (8 + 0.25 * temperature_factor)
        + staff * 12
    )


    # Increase consumption during special events
    if event:
        daily_water *= 1.08


    # Add small random variation
    daily_water *= np.random.uniform(0.95, 1.05)


    records.append({
        "Students": students,
        "Staff": staff,
        "Temperature": temperature,
        "Event": event,
        "Water Consumption": daily_water
    })




# --------------------------------
# 2. Create DataFrame
# --------------------------------


training_data = pd.DataFrame(records)


print("Dataset created successfully!")
print(training_data.head())




# --------------------------------
# 3. Select input and target
# --------------------------------


features = [
    "Students",
    "Staff",
    "Temperature",
    "Event"
]


X = training_data[features]


y = training_data["Water Consumption"]




# --------------------------------
# 4. Split training and testing data
# --------------------------------


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


print("Training records:", len(X_train))
print("Testing records:", len(X_test))




# --------------------------------
# 5. Create Linear Regression model
# --------------------------------


model = LinearRegression()




# --------------------------------
# 6. Train the model
# --------------------------------


model.fit(X_train, y_train)


print("Linear Regression model trained successfully!")




# --------------------------------
# 7. Predict using testing data
# --------------------------------


y_pred = model.predict(X_test)




# --------------------------------
# 8. Evaluate model performance
# --------------------------------


mae = mean_absolute_error(y_test, y_pred)


r2 = r2_score(y_test, y_pred)


print("Mean Absolute Error:", round(mae, 2), "litres")


print("R2 Score:", round(r2, 4))




# --------------------------------
# 9. Display model coefficients
# --------------------------------


print("\nModel Coefficients:")


for feature, coefficient in zip(features, model.coef_):
    print(feature, ":", round(coefficient, 2))


print("Intercept:", round(model.intercept_, 2))

Dataset created successfully!
   Students  Staff  Temperature  Event  Water Consumption
0      1160     34    36.299849      0       13090.521883
1      2469     94    29.481222      0       23885.131369
2       430     72    42.247746      1        6942.455251
3      1515     40    33.437038      1       17101.402706
4      1047    108    25.280729      0        9647.890390
Training records: 1200
Testing records: 300
Linear Regression model trained successfully!
Mean Absolute Error: 950.18 litres
R2 Score: 0.9644

Model Coefficients:
Students : 9.88
Staff : 13.33
Temperature : 292.56
Event : 1069.17
Intercept: -9344.24


In [ ]:
import gradio as gr
import matplotlib.pyplot as plt
import pandas as pd


# --------------------------------
# 1. Prediction function
# --------------------------------


def predict_daily_water(students, staff, temperature, special_event):


    # Validate inputs
    if students < 0 or staff < 0:
        raise gr.Error("Students and staff cannot be negative.")


    if not 0 <= temperature <= 60:
        raise gr.Error("Temperature must be between 0 and 60°C.")


    if students + staff == 0:
        raise gr.Error("At least one student or staff member is required.")


    # Prepare input for Linear Regression model
    input_data = pd.DataFrame([{
        "Students": students,
        "Staff": staff,
        "Temperature": temperature,
        "Event": int(special_event)
    }])


    # Predict daily water consumption
    prediction = model.predict(input_data)[0]


    prediction = max(0, prediction)


    # --------------------------------
    # 2. Calculate comparison scenarios
    # --------------------------------


    scenarios = {
        "Current Conditions": [students, staff, temperature, int(special_event)],
        "Temperature +5°C": [students, staff, temperature + 5, int(special_event)],
        "Students +20%": [int(students * 1.2), staff, temperature, int(special_event)],
        "Students -20%": [int(students * 0.8), staff, temperature, int(special_event)]
    }


    scenario_names = list(scenarios.keys())


    scenario_data = pd.DataFrame(
        scenarios.values(),
        columns=["Students", "Staff", "Temperature", "Event"]
    )


    scenario_predictions = model.predict(scenario_data)
    scenario_predictions = [max(0, value) for value in scenario_predictions]


    # --------------------------------
    # 3. Create Matplotlib graph
    # --------------------------------


    fig, ax = plt.subplots(figsize=(9, 5))


    bars = ax.bar(
        scenario_names,
        scenario_predictions
    )


    ax.set_title("Daily Water Consumption Prediction")
    ax.set_ylabel("Water Consumption (Litres)")
    ax.set_xlabel("School Conditions")


    ax.bar_label(
        bars,
        fmt="%.0f L",
        padding=3
    )


    plt.xticks(rotation=15)
    plt.tight_layout()


    # --------------------------------
    # 4. Display prediction results
    # --------------------------------


    result = f"""
## Predicted Daily Water Consumption


# {prediction:,.0f} Litres


### School Information
- Students Present: {students}
- Staff Present: {staff}
- Temperature: {temperature}°C
- Special Event: {"Yes" if special_event else "No"}


### AI Model
- Algorithm: Linear Regression
- Prediction Type: Daily Total
"""


    return result, fig




# --------------------------------
# 5. Create Gradio dashboard
# --------------------------------


with gr.Blocks() as demo:


    gr.Markdown(
        """
        # AI School Water Consumption Predictor


        Enter school information to estimate the total
        water consumption for the day.
        """
    )


    with gr.Row():


        students = gr.Number(
            label="Number of Students Present",
            value=2000,
            precision=0
        )


        staff = gr.Number(
            label="Number of Staff Present",
            value=100,
            precision=0
        )


    with gr.Row():


        temperature = gr.Slider(
            minimum=0,
            maximum=60,
            value=30,
            step=1,
            label="Current Temperature (°C)"
        )


        special_event = gr.Checkbox(
            label="Special Event Today",
            value=False
        )


    predict_button = gr.Button(
        "Predict Water Consumption",
        variant="primary"
    )


    output_text = gr.Markdown()


    output_graph = gr.Plot()


    predict_button.click(
        fn=predict_daily_water,
        inputs=[
            students,
            staff,
            temperature,
            special_event
        ],
        outputs=[
            output_text,
            output_graph
        ]
    )




# --------------------------------
# 6. Launch dashboard
# --------------------------------


demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://411d5240535c70d627.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
